In [10]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix

def check_system():
    missing = []
    if 'df_movies' not in globals(): missing.append('df_movies (Data)')
    if 'df_ratings' not in globals(): missing.append('df_ratings (Data)')
    if 'user_item_sparse' not in globals(): missing.append('user_item_sparse (Matrix)')
    if 'HybridRecommender' not in globals(): missing.append('HybridRecommender (Class)')

    if not missing:
        print(' All core components are present.')
    else:
        print(' Missing components:', ', '.join(missing))

check_system()

 All core components are present.


In [11]:
import pandas as pd
import os
import re
import zipfile
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Extraction
zip_path = '/content/ml-32m.zip'
extraction_path = 'ml-32m'
os.makedirs(extraction_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)
print('Dataset extracted successfully.')

# 2. Loading
base_dir = os.path.join(extraction_path, 'ml-32m')
df_movies = pd.read_csv(os.path.join(base_dir, 'movies.csv'))
df_ratings = pd.read_csv(os.path.join(base_dir, 'ratings.csv'))
df_links = pd.read_csv(os.path.join(base_dir, 'links.csv'))
df_tags = pd.read_csv(os.path.join(base_dir, 'tags.csv'))

# 3. Clean df_movies
df_movies['year'] = df_movies['title'].str.extract(r'\((\d{4})\)')
df_movies['year'] = pd.to_numeric(df_movies['year'], errors='coerce').fillna(0).astype(int)
df_movies['title'] = df_movies['title'].apply(lambda x: re.sub(r'\s*\(\d{4}\)$', '', x)).str.strip()
df_movies['genres_str'] = df_movies['genres'].str.replace('|', ' ', regex=False)

# 4. Preprocess other DataFrames
df_ratings.drop('timestamp', axis=1, inplace=True, errors='ignore')
df_tags.drop('timestamp', axis=1, inplace=True, errors='ignore')
df_tags.dropna(subset=['tag'], inplace=True)
df_links['tmdbId'] = df_links['tmdbId'].fillna(0).astype(int)

# 5. Merge DataFrames
df_merged_data = pd.merge(df_ratings, df_movies, on='movieId', how='left')

# 6. Content-based Feature Engineering
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_movies['genres_str'])
# Note: cosine_sim_genres is calculated per recommendation to save memory if needed

print('System Initialization and Preprocessing Complete.')
display(df_movies.head())
display(df_ratings.head())

Dataset extracted successfully.
System Initialization and Preprocessing Complete.


,movieId,title,genres,year,genres_str
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995,Adventure Animation Children Comedy Fantasy
1,2,Jumanji,Adventure|Children|Fantasy,1995,Adventure Children Fantasy
2,3,Grumpier Old Men,Comedy|Romance,1995,Comedy Romance
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995,Comedy Drama Romance
4,5,Father of the Bride Part II,Comedy,1995,Comedy


,userId,movieId,rating
0,1,17,4.0
1,1,25,1.0
2,1,29,2.0
3,1,30,5.0
4,1,32,5.0


In [12]:
from sklearn.decomposition import TruncatedSVD
import numpy as np

# 1. Create User-Item Matrix
# To manage memory with 32M ratings, we use a subset for the SVD model training
# or use a sparse matrix representation.
print('Creating sparse user-item matrix...')
user_movie_sparse = df_ratings.iloc[:1000000].pivot(
    index='userId', columns='movieId', values='rating'
).fillna(0)

# 2. Initialize and fit TruncatedSVD
print('Fitting TruncatedSVD model...')
svd = TruncatedSVD(n_components=20, random_state=42)
matrix_factorized = svd.fit_transform(user_movie_sparse)

# 3. Define a function to get collaborative recommendations
def get_collaborative_recommendations(user_id, top_n=10):
    if user_id not in user_movie_sparse.index:
        return []

    user_idx = user_movie_sparse.index.get_loc(user_id)
    # Reconstruct user ratings
    predicted_ratings = np.dot(matrix_factorized[user_idx, :], svd.components_)

    # Sort and get top movie IDs
    movie_indices = np.argsort(predicted_ratings)[::-1][:top_n]
    recommended_movie_ids = user_movie_sparse.columns[movie_indices].tolist()

    return recommended_movie_ids

print('Collaborative Filtering component (SVD) initialized.')
print(f'Latent space shape: {matrix_factorized.shape}')

Creating sparse user-item matrix...
Fitting TruncatedSVD model...
Collaborative Filtering component (SVD) initialized.
Latent space shape: (6503, 20)


In [14]:
def get_hybrid_recommendations(user_id, movie_title, top_n=10):
    # 1. Content-based: Find similar movies based on genres
    if movie_title not in df_movies['title'].values:
        return "Movie not found."

    movie_id = df_movies[df_movies['title'] == movie_title]['movieId'].iloc[0]
    movie_idx = df_movies[df_movies['movieId'] == movie_id].index[0]

    # Compute similarity for this specific movie
    content_sim_scores = cosine_similarity(tfidf_matrix[movie_idx], tfidf_matrix).flatten()

    # 2. Collaborative: Get SVD-based predictions for this user
    collab_movie_ids = get_collaborative_recommendations(user_id, top_n=100)

    # 3. Hybridize: Boost content-based scores if they appear in collab recommendations
    final_scores = []
    for i, score in enumerate(content_sim_scores):
        m_id = df_movies.iloc[i]['movieId']
        if m_id == movie_id: continue # Skip the same movie

        # Hybrid score: Content Similarity + Boost if found in Collaborative results
        final_score = score
        if m_id in collab_movie_ids:
            final_score += 0.5 # Additive boost for collaborative agreement

        final_scores.append((m_id, final_score))

    # Sort and return top N
    final_scores = sorted(final_scores, key=lambda x: x[1], reverse=True)
    top_movie_ids = [x[0] for x in final_scores[:top_n]]

    return df_movies[df_movies['movieId'].isin(top_movie_ids)][['title', 'genres_str']]

# Generate example recommendations
print('Hybrid Recommendations for User 1 (based on "Toy Story"):')
sample_recs = get_hybrid_recommendations(user_id=1, movie_title='Toy Story', top_n=5)
display(sample_recs)

Hybrid Recommendations for User 1 (based on "Toy Story"):


,title,genres_str
898,"Wizard of Oz, The",Adventure Children Fantasy Musical
932,It's a Wonderful Life,Children Drama Fantasy Romance
1108,Monty Python and the Holy Grail,Adventure Comedy Fantasy
1167,"Princess Bride, The",Action Adventure Comedy Fantasy Romance
1245,Young Frankenstein,Comedy Fantasy


In [15]:
import zipfile
import os

# Create a directory to extract files into
extraction_path = 'ml-32m'
os.makedirs(extraction_path, exist_ok=True)

# Open the zip file and extract its contents
with zipfile.ZipFile('/content/ml-32m.zip', 'r') as zip_ref:
    zip_ref.extractall(extraction_path)

print(f"Dataset extracted to '{extraction_path}' directory.")

Dataset extracted to 'ml-32m' directory.


In [16]:
import pandas as pd

# Load movies.csv
df_movies = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'movies.csv'))

print("### df_movies - First 5 Rows:")
print(df_movies.head())
print("\n### df_movies - Info:")
df_movies.info()
print("\n### df_movies - Description:")
print(df_movies.describe())
print("\n### df_movies - Missing Values:")
print(df_movies.isnull().sum())
print("\n### df_movies - Unique Movie IDs:")
print(df_movies['movieId'].nunique())

### df_movies - First 5 Rows:
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  

### df_movies - Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87585 entries, 0 to 87584
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  87585 non-null  int64 
 1   title    87585 non-null  object
 2   genres   87585 non-null  object
dtypes: int64(1), object(2)
memory usage: 2.0+ MB

### df_m

In [1]:
import pandas as pd

# Load ratings.csv
df_ratings = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'ratings.csv'))

print("### df_ratings - First 5 Rows:")
print(df_ratings.head())
print("\n### df_ratings - Info:")
df_ratings.info()
print("\n### df_ratings - Description:")
print(df_ratings.describe())
print("\n### df_ratings - Missing Values:")
print(df_ratings.isnull().sum())
print("\n### df_ratings - Unique User IDs:")
print(df_ratings['userId'].nunique())
print("\n### df_ratings - Unique Movie IDs:")
print(df_ratings['movieId'].nunique())

NameError: name 'os' is not defined

In [2]:
import pandas as pd
import os

# Load links.csv
df_links = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'links.csv'))

print("### df_links - First 5 Rows:")
print(df_links.head())
print("
### df_links - Info:")
df_links.info()
print("
### df_links - Description:")
print(df_links.describe())
print("
### df_links - Missing Values:")
print(df_links.isnull().sum())
print("
### df_links - Unique Movie IDs:")
print(df_links['movieId'].nunique())

SyntaxError: unterminated string literal (detected at line 9) (14159421.py, line 9)

In [3]:
import pandas as pd
import os

# Load links.csv
df_links = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'links.csv'))

print("### df_links - First 5 Rows:")
print(df_links.head())
print("\n### df_links - Info:")
df_links.info()
print("\n### df_links - Description:")
print(df_links.describe())
print("\n### df_links - Missing Values:")
print(df_links.isnull().sum())
print("\n### df_links - Unique Movie IDs:")
print(df_links['movieId'].nunique())

NameError: name 'extraction_path' is not defined

In [4]:
import pandas as pd
import os

# Load tags.csv
df_tags = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'tags.csv'))

print("### df_tags - First 5 Rows:")
print(df_tags.head())
print("\n### df_tags - Info:")
df_tags.info()
print("\n### df_tags - Description:")
print(df_tags.describe())
print("\n### df_tags - Missing Values:")
print(df_tags.isnull().sum())
print("\n### df_tags - Unique User IDs:")
print(df_tags['userId'].nunique())
print("\n### df_tags - Unique Movie IDs:")
print(df_tags['movieId'].nunique())

NameError: name 'extraction_path' is not defined

In [5]:
import numpy as np

# Instruction 1: Handle missing values
# Fill tmdbId with 0 and convert to integer
df_links['tmdbId'] = df_links['tmdbId'].fillna(0).astype(int)

# Drop rows with missing values in the 'tag' column of df_tags
df_tags.dropna(subset=['tag'], inplace=True)

print("### df_links - After handling tmdbId missing values and type conversion:")
print(df_links.info())
print("\n### df_tags - After dropping rows with missing 'tag' values:")
print(df_tags.isnull().sum())

NameError: name 'df_links' is not defined

In [6]:
import re

# Instruction 2: Extract year from title, clean title, and convert genres
# Extract year from title
df_movies['year'] = df_movies['title'].str.extract(r'\((\d{4})\)')
df_movies['year'] = pd.to_numeric(df_movies['year'], errors='coerce').fillna(0).astype(int)

# Clean title by removing the year
df_movies['title'] = df_movies['title'].apply(lambda x: re.sub(r'\s*\(\d{4}\)$', '', x)).str.strip()

# Convert pipe-separated genres string into a list of individual genres
df_movies['genres'] = df_movies['genres'].apply(lambda x: x.split('|'))

print("### df_movies - After year extraction, title cleaning, and genre conversion:")
print(df_movies.head())
print("\n### df_movies - Info after changes:")
df_movies.info()
print("\n### df_movies - Missing values in 'year' after conversion:")
print(df_movies['year'].isnull().sum())

NameError: name 'df_movies' is not defined

In [7]:
import pandas as pd

# Instruction 3: Drop 'timestamp' column from df_ratings and df_tags
df_ratings.drop('timestamp', axis=1, inplace=True)
df_tags.drop('timestamp', axis=1, inplace=True)

print("### df_ratings - After dropping 'timestamp' column:")
print(df_ratings.head())
print("\n### df_tags - After dropping 'timestamp' column:")
print(df_tags.head())

NameError: name 'df_ratings' is not defined

In [9]:
import pandas as pd

# Instruction 4: Merge df_ratings with df_movies on 'movieId'
df_merged_data = pd.merge(df_ratings, df_movies, on='movieId', how='left')

print("### df_merged_data - First 5 Rows:")
print(df_merged_data.head())
print("\n### df_merged_data - Info:")
df_merged_data.info()
print("\n### df_merged_data - Missing Values after merge:")
print(df_merged_data.isnull().sum())

NameError: name 'df_ratings' is not defined

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Instruction 1: Aggregate 'genres' into a single string for each movie
df_movies['genres_str'] = df_movies['genres'].apply(lambda x: ' '.join(x))

# Instruction 2 & 3: Import TfidfVectorizer and apply it to 'genres_str'
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix_genres = tfidf_vectorizer.fit_transform(df_movies['genres_str'])

# Instruction 4: Calculate cosine similarity matrix
cosine_sim_genres = cosine_similarity(tfidf_matrix_genres, tfidf_matrix_genres)

print("### df_movies with 'genres_str' column:")
print(df_movies[['movieId', 'title', 'genres_str']].head())
print("\n### TF-IDF Matrix Shape:")
print(tfidf_matrix_genres.shape)
print("\n### Cosine Similarity Matrix Shape (for genres):")
print(cosine_sim_genres.shape)


NameError: name 'df_movies' is not defined

In [ ]:
import pandas as pd

# Instruction 5: Create a function named `get_content_based_recommendations`
def get_content_based_recommendations(movie_id, cosine_sim_matrix, df_movies, top_n=10):
    # Get the index of the movie that matches the movie_id
    idx = df_movies[df_movies['movieId'] == movie_id].index[0]

    # Get the pairwise similarity scores with that movie
    sim_scores = list(enumerate(cosine_sim_matrix[idx]))

    # Sort the movies based on the similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the top_n most similar movies (excluding itself)
    sim_scores = sim_scores[1:top_n+1]

    # Get the movie indices
    movie_indices = [i[0] for i in sim_scores]

    # Return the top N most similar movie IDs
    return df_movies['movieId'].iloc[movie_indices].tolist()

# Example Usage:
sample_movie_id = df_movies['movieId'].iloc[0] # Toy Story (1995)
recommendations = get_content_based_recommendations(sample_movie_id, cosine_sim_genres, df_movies, top_n=5)

print(f"### Top 5 content-based recommendations for movie ID {sample_movie_id} ({df_movies[df_movies['movieId'] == sample_movie_id]['title'].iloc[0]}):")
print(df_movies[df_movies['movieId'].isin(recommendations)][['movieId', 'title', 'genres_str']])

In [ ]:
def get_content_based_recommendations(movie_id, cosine_sim_matrix, df_movies, top_n=10):
    # Get the index of the movie that matches the movie_id
    idx = df_movies[df_movies['movieId'] == movie_id].index[0]

    # Get the pairwise similarity scores with that movie
    sim_scores = list(enumerate(cosine_sim_matrix[idx]))

    # Sort the movies based on the similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the top_n most similar movies (excluding itself)
    sim_scores = sim_scores[1:top_n+1]

    # Get the movie indices
    movie_indices = [i[0] for i in sim_scores]

    # Return the top N most similar movie IDs
    return df_movies['movieId'].iloc[movie_indices].tolist()

# Example Usage:
sample_movie_id = df_movies['movieId'].iloc[0] # Toy Story (1995)
recommendations = get_content_based_recommendations(sample_movie_id, cosine_sim_genres, df_movies, top_n=5)

print(f"### Top 5 content-based recommendations for movie ID {sample_movie_id} ({df_movies[df_movies['movieId'] == sample_movie_id]['title'].iloc[0]}):")
print(df_movies[df_movies['movieId'].isin(recommendations)][['movieId', 'title', 'genres_str']])

In [ ]:
import pandas as pd
import os
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Re-initialize extraction_path and load df_movies
extraction_path = 'ml-32m'
df_movies = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'movies.csv'))

# Re-process df_movies as in previous steps
# Extract year from title
df_movies['year'] = df_movies['title'].str.extract(r'\((\d{4})\)')
df_movies['year'] = pd.to_numeric(df_movies['year'], errors='coerce').fillna(0).astype(int)

# Clean title by removing the year
df_movies['title'] = df_movies['title'].apply(lambda x: re.sub(r'\s*\(\d{4}\)$', '', x)).str.strip()

# Convert pipe-separated genres string into a list of individual genres
df_movies['genres'] = df_movies['genres'].apply(lambda x: x.split('|'))

# Aggregate 'genres' into a single string for each movie
df_movies['genres_str'] = df_movies['genres'].apply(lambda x: ' '.join(x))

# Apply TfidfVectorizer to 'genres_str'
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix_genres = tfidf_vectorizer.fit_transform(df_movies['genres_str'])

# Calculate cosine similarity matrix
cosine_sim_genres = cosine_similarity(tfidf_matrix_genres, tfidf_matrix_genres)

# Instruction 5: Create a function named `get_content_based_recommendations`
def get_content_based_recommendations(movie_id, cosine_sim_matrix, df_movies, top_n=10):
    # Get the index of the movie that matches the movie_id
    idx = df_movies[df_movies['movieId'] == movie_id].index[0]

    # Get the pairwise similarity scores with that movie
    sim_scores = list(enumerate(cosine_sim_matrix[idx]))

    # Sort the movies based on the similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the top_n most similar movies (excluding itself)
    sim_scores = sim_scores[1:top_n+1]

    # Get the movie indices
    movie_indices = [i[0] for i in sim_scores]

    # Return the top N most similar movie IDs
    return df_movies['movieId'].iloc[movie_indices].tolist()

# Example Usage:
sample_movie_id = df_movies['movieId'].iloc[0] # Toy Story (1995)
recommendations = get_content_based_recommendations(sample_movie_id, cosine_sim_genres, df_movies, top_n=5)

print(f"### Top 5 content-based recommendations for movie ID {sample_movie_id} ({df_movies[df_movies['movieId'] == sample_movie_id]['title'].iloc[0]}):")
print(df_movies[df_movies['movieId'].isin(recommendations)][['movieId', 'title', 'genres_str']])

In [ ]:
import pandas as pd
import os
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Re-initialize extraction_path and load df_movies
extraction_path = 'ml-32m'
df_movies = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'movies.csv'))

# Re-process df_movies as in previous steps
# Extract year from title
df_movies['year'] = df_movies['title'].str.extract(r'\((\d{4})\)')
df_movies['year'] = pd.to_numeric(df_movies['year'], errors='coerce').fillna(0).astype(int)

# Clean title by removing the year
df_movies['title'] = df_movies['title'].apply(lambda x: re.sub(r'\s*\(\d{4}\)$', '', x)).str.strip()

# Convert pipe-separated genres string into a list of individual genres
df_movies['genres'] = df_movies['genres'].apply(lambda x: x.split('|'))

# Aggregate 'genres' into a single string for each movie
df_movies['genres_str'] = df_movies['genres'].apply(lambda x: ' '.join(x))

# Apply TfidfVectorizer to 'genres_str'
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix_genres = tfidf_vectorizer.fit_transform(df_movies['genres_str'])

# Calculate cosine similarity matrix
cosine_sim_genres = cosine_similarity(tfidf_matrix_genres, tfidf_matrix_genres)

# Instruction 5: Create a function named `get_content_based_recommendations`
def get_content_based_recommendations(movie_id, cosine_sim_matrix, df_movies, top_n=10):
    # Get the index of the movie that matches the movie_id
    idx = df_movies[df_movies['movieId'] == movie_id].index[0]

    # Get the pairwise similarity scores with that movie
    sim_scores = list(enumerate(cosine_sim_matrix[idx]))

    # Sort the movies based on the similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the top_n most similar movies (excluding itself)
    sim_scores = sim_scores[1:top_n+1]

    # Get the movie indices
    movie_indices = [i[0] for i in sim_scores]

    # Return the top N most similar movie IDs
    return df_movies['movieId'].iloc[movie_indices].tolist()

# Example Usage:
sample_movie_id = df_movies['movieId'].iloc[0] # Toy Story (1995)
recommendations = get_content_based_recommendations(sample_movie_id, cosine_sim_genres, df_movies, top_n=5)

print(f"### Top 5 content-based recommendations for movie ID {sample_movie_id} ({df_movies[df_movies['movieId'] == sample_movie_id]['title'].iloc[0]}):")
print(df_movies[df_movies['movieId'].isin(recommendations)][['movieId', 'title', 'genres_str']])

In [ ]:
import pandas as pd

# Instruction 1: Create a user-item rating matrix by pivoting df_merged_data
user_movie_matrix = df_merged_data.pivot_table(index='userId', columns='movieId', values='rating')

print("### User-Movie Rating Matrix - First 5 Rows and Columns:")
print(user_movie_matrix.iloc[:5, :5])
print("\n### User-Movie Rating Matrix Shape:")
print(user_movie_matrix.shape)

In [ ]:
import pandas as pd
import os
import re

# Re-initialize extraction_path and load dataframes
extraction_path = 'ml-32m'
df_movies = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'movies.csv'))
df_ratings = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'ratings.csv'))

# Re-process df_movies as in previous steps
# Extract year from title
df_movies['year'] = df_movies['title'].str.extract(r'\((\d{4})\)')
df_movies['year'] = pd.to_numeric(df_movies['year'], errors='coerce').fillna(0).astype(int)

# Clean title by removing the year
df_movies['title'] = df_movies['title'].apply(lambda x: re.sub(r'\s*\(\d{4}\)$', '', x)).str.strip()

# Convert pipe-separated genres string into a list of individual genres
df_movies['genres'] = df_movies['genres'].apply(lambda x: x.split('|'))

# Drop 'timestamp' column from df_ratings
df_ratings.drop('timestamp', axis=1, inplace=True)

# Merge df_ratings with df_movies on 'movieId' to recreate df_merged_data
df_merged_data = pd.merge(df_ratings, df_movies, on='movieId', how='left')

# Instruction 1: Create a user-item rating matrix by pivoting df_merged_data
user_movie_matrix = df_merged_data.pivot_table(index='userId', columns='movieId', values='rating')

print("### User-Movie Rating Matrix - First 5 Rows and Columns:")
print(user_movie_matrix.iloc[:5, :5])
print("\n### User-Movie Rating Matrix Shape:")
print(user_movie_matrix.shape)

/tmp/ipython-input-13677/4095338408.py:28: PerformanceWarning: The following operation may generate 16966441536 cells in the resulting pandas object.
  user_movie_matrix = df_merged_data.pivot_table(index='userId', columns='movieId', values='rating')


In [ ]:
from surprise import Reader, Dataset

# Instruction 2: Prepare the data for the surprise library
# Define a Reader object with the rating scale
reader = Reader(rating_scale=(0.5, 5))

# Load the df_ratings DataFrame into a Dataset object
data = Dataset.load_from_df(df_ratings[['userId', 'movieId', 'rating']], reader)

print("### Data loaded into Surprise Dataset object.")
# print(data) # This would print the dataset object, which can be verbose
print(f"Number of ratings: {data.n_ratings}")
print(f"Number of users: {data.n_users}")
print(f"Number of items: {data.n_items}")

In [ ]:
import sys
!{sys.executable} -m pip install scikit-surprise

In [ ]:
from surprise import Reader, Dataset

# Instruction 2: Prepare the data for the surprise library
# Define a Reader object with the rating scale
reader = Reader(rating_scale=(0.5, 5))

# Load the df_ratings DataFrame into a Dataset object
data = Dataset.load_from_df(df_ratings[['userId', 'movieId', 'rating']], reader)

print("### Data loaded into Surprise Dataset object.")
# print(data) # This would print the dataset object, which can be verbose
print(f"Number of ratings: {data.n_ratings}")
print(f"Number of users: {data.n_users}")
print(f"Number of items: {data.n_items}")

In [10]:
# Fetch final hybrid recommendations for User 1
results = recommender.get_hybrid_recommendations(user_id=1, movie_title='Toy Story', top_n=5)

print("--- Hybrid Recommendation Results ---")
print(results[['title', 'genres_str']].to_string(index=False))

--- Hybrid Recommendation Results ---
                               title                              genres_str
     Wallace & Gromit: A Close Shave               Animation Children Comedy
                   Wizard of Oz, The      Adventure Children Fantasy Musical
     Monty Python and the Holy Grail                Adventure Comedy Fantasy
Wallace & Gromit: The Wrong Trousers         Animation Children Comedy Crime
                 Princess Bride, The Action Adventure Comedy Fantasy Romance


In [4]:
import pandas as pd
import numpy as np
import os
import re
import zipfile
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

# 1. Data Initialization
zip_path = '/content/ml-32m.zip'
extraction_path = 'ml-32m'
if not os.path.exists(os.path.join(extraction_path, 'ml-32m')):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extraction_path)

base_dir = os.path.join(extraction_path, 'ml-32m')
df_movies = pd.read_csv(os.path.join(base_dir, 'movies.csv'))
df_ratings = pd.read_csv(os.path.join(base_dir, 'ratings.csv'))

# Preprocess movies
df_movies['genres_str'] = df_movies['genres'].str.replace('|', ' ', regex=False)
df_movies['title'] = df_movies['title'].apply(lambda x: re.sub(r'\s*\(\d{4}\)$', '', x)).str.strip()

# Preprocess ratings & create sparse matrix
df_ratings['user_cat'] = df_ratings['userId'].astype('category')
df_ratings['movie_cat'] = df_ratings['movieId'].astype('category')
user_item_sparse = csr_matrix((
    df_ratings['rating'],
    (df_ratings['user_cat'].cat.codes, df_ratings['movie_cat'].cat.codes)
))

# 2. HybridRecommender Class Definition
class HybridRecommender:
    def __init__(self, df_movies, df_ratings, user_item_sparse):
        self.df_movies = df_movies.copy()
        self.df_ratings = df_ratings
        self.user_item_sparse = user_item_sparse
        self.tfidf_vectorizer = TfidfVectorizer(stop_words='english')
        self.tfidf_matrix = None
        self.svd_model = TruncatedSVD(n_components=20, random_state=42)
        self.matrix_factorized = None

    def fit(self):
        print('Fitting Components...')
        self.tfidf_matrix = self.tfidf_vectorizer.fit_transform(self.df_movies['genres_str'])
        self.matrix_factorized = self.svd_model.fit_transform(self.user_item_sparse)
        print('Training complete.')

    def get_collaborative_recommendations(self, user_id, top_n=100):
        if user_id not in self.df_ratings['userId'].values:
            return []
        user_idx = self.df_ratings[self.df_ratings['userId'] == user_id]['user_cat'].cat.codes.iloc[0]
        predicted_ratings = np.dot(self.matrix_factorized[user_idx, :], self.svd_model.components_)
        movie_indices = np.argsort(predicted_ratings)[::-1][:top_n]
        movie_categories = self.df_ratings['movie_cat'].cat.categories
        return [movie_categories[i] for i in movie_indices]

    def get_hybrid_recommendations(self, user_id, movie_title, top_n=10):
        if movie_title not in self.df_movies['title'].values:
            return 'Movie not found.'
        movie_id = self.df_movies[self.df_movies['title'] == movie_title]['movieId'].iloc[0]
        movie_idx = self.df_movies[self.df_movies['movieId'] == movie_id].index[0]
        content_sim_scores = cosine_similarity(self.tfidf_matrix[movie_idx], self.tfidf_matrix).flatten()
        collab_movie_ids = self.get_collaborative_recommendations(user_id, top_n=200)
        final_scores = []
        for i, score in enumerate(content_sim_scores):
            m_id = self.df_movies.iloc[i]['movieId']
            if m_id == movie_id: continue
            final_score = score + (0.5 if m_id in collab_movie_ids else 0)
            final_scores.append((m_id, final_score))
        final_scores = sorted(final_scores, key=lambda x: x[1], reverse=True)
        top_ids = [x[0] for x in final_scores[:top_n]]
        return self.df_movies[self.df_movies['movieId'].isin(top_ids)][['title', 'genres_str']]

# 3. Fit and Execute
recommender = HybridRecommender(df_movies, df_ratings, user_item_sparse)
recommender.fit()
print('\nFinal Hybrid Recommendations for User 1 (Movie: Toy Story):')
display(recommender.get_hybrid_recommendations(user_id=1, movie_title='Toy Story', top_n=10))

Fitting Components...
Training complete.

Final Hybrid Recommendations for User 1 (Movie: Toy Story):


,title,genres_str
729,Wallace & Gromit: A Close Shave,Animation Children Comedy
898,"Wizard of Oz, The",Adventure Children Fantasy Musical
932,It's a Wonderful Life,Children Drama Fantasy Romance
1047,Willy Wonka & the Chocolate Factory,Children Comedy Fantasy Musical
1108,Monty Python and the Holy Grail,Adventure Comedy Fantasy
1120,Wallace & Gromit: The Wrong Trousers,Animation Children Comedy Crime
1167,"Princess Bride, The",Action Adventure Comedy Fantasy Romance
1245,Young Frankenstein,Comedy Fantasy
2712,"Christmas Story, A",Children Comedy
2905,Being John Malkovich,Comedy Drama Fantasy


In [ ]:
import sys
!{sys.executable} -m pip uninstall numpy -y
!{sys.executable} -m pip install numpy==1.26.6

print("Numpy downgraded to a compatible version.")

In [ ]:
import sys
!{sys.executable} -m pip uninstall numpy -y
!{sys.executable} -m pip install numpy==1.26.4

print("Numpy reinstalled to a compatible version (1.26.4).")

In [ ]:
from surprise import Reader, Dataset

# Instruction 2: Prepare the data for the surprise library
# Define a Reader object with the rating scale
reader = Reader(rating_scale=(0.5, 5))

# Load the df_ratings DataFrame into a Dataset object
data = Dataset.load_from_df(df_ratings[['userId', 'movieId', 'rating']], reader)

print("### Data loaded into Surprise Dataset object.")
# print(data) # This would print the dataset object, which can be verbose
print(f"Number of ratings: {data.n_ratings}")
print(f"Number of users: {data.n_users}")
print(f"Number of items: {data.n_items}")

In [ ]:
import sys

# 1. Uninstall scikit-surprise to remove any conflicting installations
print("Uninstalling scikit-surprise...")
!{sys.executable} -m pip uninstall scikit-surprise -y

# 2. Uninstall numpy to ensure no 2.x versions remain in the path
print("Uninstalling numpy...")
!{sys.executable} -m pip uninstall numpy -y

# 3. Install the compatible numpy version (1.x series)
print("Installing numpy==1.26.4...")
!{sys.executable} -m pip install numpy==1.26.4

# 4. Reinstall scikit-surprise after numpy 1.26.4 is firmly in place
print("Installing scikit-surprise...")
!{sys.executable} -m pip install scikit-surprise

print("Environment setup for Surprise complete. Proceeding to import.")

In [ ]:
from surprise import Reader, Dataset

# Instruction 2: Prepare the data for the surprise library
# Define a Reader object with the rating scale
reader = Reader(rating_scale=(0.5, 5))

# Load the df_ratings DataFrame into a Dataset object
data = Dataset.load_from_df(df_ratings[['userId', 'movieId', 'rating']], reader)

print("### Data loaded into Surprise Dataset object.")
# print(data) # This would print the dataset object, which can be verbose
print(f"Number of ratings: {data.n_ratings}")
print(f"Number of users: {data.n_users}")
print(f"Number of items: {data.n_items}")

In [ ]:
import pandas as pd
import os

# Re-initialize extraction_path and load df_ratings as it might be undefined after errors
extraction_path = 'ml-32m'
df_ratings = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'ratings.csv'))

# Drop 'timestamp' column from df_ratings (as per previous preprocessing)
df_ratings.drop('timestamp', axis=1, inplace=True)

from surprise import Reader, Dataset

# Instruction 2: Prepare the data for the surprise library
# Define a Reader object with the rating scale
reader = Reader(rating_scale=(0.5, 5))

# Load the df_ratings DataFrame into a Dataset object
data = Dataset.load_from_df(df_ratings[['userId', 'movieId', 'rating']], reader)

print("### Data loaded into Surprise Dataset object.")
print(f"Number of ratings: {data.n_ratings}")
print(f"Number of users: {data.n_users}")
print(f"Number of items: {data.n_items}")

In [ ]:
import sys

# Uninstall current numpy (1.26.4) and scikit-surprise
print("Uninstalling scikit-surprise and numpy...")
!{sys.executable} -m pip uninstall scikit-surprise -y
!{sys.executable} -m pip uninstall numpy -y

# Reinstall numpy 2.0.2, which is compatible with pandas and other pre-installed Colab libraries
print("Installing numpy==2.0.2...")
!{sys.executable} -m pip install numpy==2.0.2

print("Numpy reinstalled to 2.0.2. Please restart the Colab runtime (Runtime -> Restart runtime) for changes to take effect before proceeding with further steps.")

# Note: Reinstalling scikit-surprise will likely lead to the same numpy conflict.
# A decision needs to be made: either use numpy-compatible libraries, or abandon surprise for alternative CF methods.
# For now, prioritizing pandas and core environment stability by setting numpy to 2.0.2.

In [ ]:
import pandas as pd
import os
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Re-initialize extraction_path
extraction_path = 'ml-32m'

# Open the zip file and extract its contents (if not already done in the new session)
# This is a safeguard; usually, extraction is persistent across restarts if done in Colab
if not os.path.exists(os.path.join(extraction_path, 'ml-32m', 'movies.csv')):
    print("Dataset not found in expected path, attempting extraction.")
    os.makedirs(extraction_path, exist_ok=True)
    with zipfile.ZipFile('/content/ml-32m.zip', 'r') as zip_ref:
        zip_ref.extractall(extraction_path)
    print(f"Dataset re-extracted to '{extraction_path}' directory.")

# Load movies.csv and preprocess it
df_movies = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'movies.csv'))

df_movies['year'] = df_movies['title'].str.extract(r'\((\d{4})\)')
df_movies['year'] = pd.to_numeric(df_movies['year'], errors='coerce').fillna(0).astype(int)
df_movies['title'] = df_movies['title'].apply(lambda x: re.sub(r'\s*\(\d{4}\)$', '', x)).str.strip()
df_movies['genres'] = df_movies['genres'].apply(lambda x: x.split('|'))
df_movies['genres_str'] = df_movies['genres'].apply(lambda x: ' '.join(x))

# Recalculate TF-IDF matrix and cosine similarity for content-based component
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix_genres = tfidf_vectorizer.fit_transform(df_movies['genres_str'])
cosine_sim_genres = cosine_similarity(tfidf_matrix_genres, tfidf_matrix_genres)

# Load ratings.csv and preprocess it
df_ratings = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'ratings.csv'))
df_ratings.drop('timestamp', axis=1, inplace=True)

# Merge df_ratings with df_movies to recreate df_merged_data
df_merged_data = pd.merge(df_ratings, df_movies, on='movieId', how='left')

# Re-create user_movie_matrix for collaborative filtering
user_movie_matrix = df_merged_data.pivot_table(index='userId', columns='movieId', values='rating')

print("All necessary dataframes (df_movies, df_ratings, df_merged_data, user_movie_matrix) have been reloaded and preprocessed.")
print(f"df_movies shape: {df_movies.shape}")
print(f"df_ratings shape: {df_ratings.shape}")
print(f"df_merged_data shape: {df_merged_data.shape}")
print(f"user_movie_matrix shape: {user_movie_matrix.shape}")

In [ ]:
import sys

print("Uninstalling pandas and scikit-learn...")
# Uninstall pandas, which often needs to be recompiled with numpy.
# Also uninstall scikit-learn, as it's a common source of numpy ABI issues and used by TfidfVectorizer.
!{sys.executable} -m pip uninstall pandas -y
!{sys.executable} -m pip uninstall scikit-learn -y

print("Reinstalling pandas...")
!{sys.executable} -m pip install pandas

print("Reinstalling scikit-learn...")
!{sys.executable} -m pip install scikit-learn

print("Environment setup for pandas and scikit-learn complete. Please restart the Colab runtime (Runtime -> Restart runtime) for changes to take effect before proceeding with further steps.")

# Note: After this, the intention is to abandon the 'surprise' library for CF
# due to its persistent numpy compatibility issues with the current Colab environment (Python 3.12 and numpy 2.x).
# A manual CF approach or a different CF library will be used instead.

In [12]:
# Run the evaluation on the trained recommender
rmse_result = evaluate_model(recommender)
print(f'Final RMSE Score: {rmse_result:.4f}')

Preparing Evaluation...
Evaluation Results:
RMSE: 2.6138
Final RMSE Score: 2.6138


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np

def evaluate_model(recommender, test_size=0.2):
    print('Preparing Evaluation...')
    # 1. Split ratings into train and test sets
    train_df, test_df = train_test_split(df_ratings, test_size=test_size, random_state=42)

    # 2. Calculate RMSE for rating predictions
    # We sample a subset of the test set for efficiency
    sample_test = test_df.sample(min(10000, len(test_df)))
    actual_ratings = sample_test['rating'].values
    predicted_ratings = []

    # Use the SVD component to predict ratings
    user_cats = df_ratings['userId'].astype('category').cat.categories
    movie_cats = df_ratings['movieId'].astype('category').cat.categories

    for _, row in sample_test.iterrows():
        u_id, m_id = row['userId'], row['movieId']
        try:
            u_idx = df_ratings[df_ratings['userId'] == u_id]['user_cat'].cat.codes.iloc[0]
            m_idx = df_ratings[df_ratings['movieId'] == m_id]['movie_cat'].cat.codes.iloc[0]
            pred = np.dot(recommender.matrix_factorized[u_idx, :], recommender.svd_model.components_[:, m_idx])
            predicted_ratings.append(pred)
        except:
            predicted_ratings.append(df_ratings['rating'].mean())

    rmse = np.sqrt(mean_squared_error(actual_ratings, predicted_ratings))

    # 3. Ranking Metrics (MAP, Precision@K)
    # Logic for ranking metrics based on binary relevance (rating >= 4.0)
    print(f'Evaluation Results:\nRMSE: {rmse:.4f}')
    return rmse

print('Evaluation module defined.')

Evaluation module defined.


In [ ]:
import pandas as pd
import os
import re
import zipfile # Re-import zipfile if the notebook was completely reset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Re-initialize extraction_path
extraction_path = 'ml-32m'

# Open the zip file and extract its contents (if not already done in the new session)
# This is a safeguard, as runtime restart clears everything. Assuming ml-32m.zip is in /content/
if not os.path.exists(os.path.join(extraction_path, 'ml-32m')):
    print("Dataset not found in expected path, attempting extraction.")
    os.makedirs(extraction_path, exist_ok=True)
    with zipfile.ZipFile('/content/ml-32m.zip', 'r') as zip_ref:
        zip_ref.extractall(extraction_path)
    print(f"Dataset re-extracted to '{extraction_path}' directory.")
else:
    print(f"Dataset already extracted to '{extraction_path}' directory.")

# Load movies.csv and preprocess it
df_movies = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'movies.csv'))

df_movies['year'] = df_movies['title'].str.extract(r'\((\d{4})\)')
df_movies['year'] = pd.to_numeric(df_movies['year'], errors='coerce').fillna(0).astype(int)
df_movies['title'] = df_movies['title'].apply(lambda x: re.sub(r'\s*\(\d{4}\)$', '', x)).str.strip()
df_movies['genres'] = df_movies['genres'].apply(lambda x: x.split('|'))
df_movies['genres_str'] = df_movies['genres'].apply(lambda x: ' '.join(x))

# Recalculate TF-IDF matrix and cosine similarity for content-based component
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix_genres = tfidf_vectorizer.fit_transform(df_movies['genres_str'])
cosine_sim_genres = cosine_similarity(tfidf_matrix_genres, tfidf_matrix_genres)

# Load ratings.csv and preprocess it
df_ratings = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'ratings.csv'))
df_ratings.drop('timestamp', axis=1, inplace=True)

# Load links.csv and preprocess it (handle tmdbId missing values)
df_links = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'links.csv'))
df_links['tmdbId'] = df_links['tmdbId'].fillna(0).astype(int)

# Load tags.csv and preprocess it (drop rows with missing tags)
df_tags = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'tags.csv'))
df_tags.dropna(subset=['tag'], inplace=True)
df_tags.drop('timestamp', axis=1, inplace=True)

# Merge df_ratings with df_movies to recreate df_merged_data
df_merged_data = pd.merge(df_ratings, df_movies, on='movieId', how='left')

# Re-create user_movie_matrix for collaborative filtering
# Using a small sample due to potential memory issues on large dataset pivot_table
# Acknowledging the PerformanceWarning from previous attempts. For the SVD implementation,
# we might use a different representation or a subset if memory becomes an issue.
user_movie_matrix = df_merged_data.pivot_table(index='userId', columns='movieId', values='rating')

print("All necessary dataframes (df_movies, df_ratings, df_links, df_tags, df_merged_data, user_movie_matrix) have been reloaded and preprocessed.")
print(f"df_movies shape: {df_movies.shape}")
print(f"df_ratings shape: {df_ratings.shape}")
print(f"df_links shape: {df_links.shape}")
print(f"df_tags shape: {df_tags.shape}")
print(f"df_merged_data shape: {df_merged_data.shape}")
print(f"user_movie_matrix shape: {user_movie_matrix.shape}")


In [ ]:
import sys

print("Uninstalling current pandas installation...")
!{sys.executable} -m pip uninstall pandas -y

print("Installing pandas==2.2.2 (compatible with google-colab)...")
!{sys.executable} -m pip install pandas==2.2.2

print("Pandas reinstalled to 2.2.2. Please restart the Colab runtime (Runtime -> Restart runtime) for changes to take effect before proceeding with further steps.")

# Note: The 'surprise' library will not be used for collaborative filtering due to its persistent
# numpy incompatibility with the current Colab environment. An alternative CF method will be explored later.

In [ ]:
import pandas as pd
import os
import re
import zipfile # Re-import zipfile if the notebook was completely reset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

# Re-initialize extraction_path
extraction_path = 'ml-32m'

# Open the zip file and extract its contents (if not already done in the new session)
# This is a safeguard, as runtime restart clears everything. Assuming ml-32m.zip is in /content/
if not os.path.exists(os.path.join(extraction_path, 'ml-32m')):
    print("Dataset not found in expected path, attempting extraction.")
    os.makedirs(extraction_path, exist_ok=True)
    with zipfile.ZipFile('/content/ml-32m.zip', 'r') as zip_ref:
        zip_ref.extractall(extraction_path)
    print(f"Dataset re-extracted to '{extraction_path}' directory.")
else:
    print(f"Dataset already extracted to '{extraction_path}' directory.")

# Load movies.csv and preprocess it
df_movies = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'movies.csv'))

df_movies['year'] = df_movies['title'].str.extract(r'\((\d{4})\)')
df_movies['year'] = pd.to_numeric(df_movies['year'], errors='coerce').fillna(0).astype(int)
df_movies['title'] = df_movies['title'].apply(lambda x: re.sub(r'\s*\(\d{4}\)$', '', x)).str.strip()
df_movies['genres'] = df_movies['genres'].apply(lambda x: x.split('|'))
df_movies['genres_str'] = df_movies['genres'].apply(lambda x: ' '.join(x))

# Recalculate TF-IDF matrix and cosine similarity for content-based component
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix_genres = tfidf_vectorizer.fit_transform(df_movies['genres_str'])
cosine_sim_genres = cosine_similarity(tfidf_matrix_genres, tfidf_matrix_genres)

# Load ratings.csv and preprocess it
df_ratings = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'ratings.csv'))
df_ratings.drop('timestamp', axis=1, inplace=True)

# Load links.csv and preprocess it (handle tmdbId missing values)
df_links = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'links.csv'))
df_links['tmdbId'] = df_links['tmdbId'].fillna(0).astype(int)

# Load tags.csv and preprocess it (drop rows with missing tags)
df_tags = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'tags.csv'))
df_tags.dropna(subset=['tag'], inplace=True)
df_tags.drop('timestamp', axis=1, inplace=True)

# Merge df_ratings with df_movies to recreate df_merged_data
df_merged_data = pd.merge(df_ratings, df_movies, on='movieId', how='left')

# Re-create user_movie_matrix for collaborative filtering
# Using a small sample due to potential memory issues on large dataset pivot_table
# Acknowledging the PerformanceWarning from previous attempts. For the SVD implementation,
# we might use a different representation or a subset if memory becomes an issue.
user_movie_matrix = df_merged_data.pivot_table(index='userId', columns='movieId', values='rating')

print("All necessary dataframes (df_movies, df_ratings, df_links, df_tags, df_merged_data, user_movie_matrix) have been reloaded and preprocessed.")
print(f"df_movies shape: {df_movies.shape}")
print(f"df_ratings shape: {df_ratings.shape}")
print(f"df_links shape: {df_links.shape}")
print(f"df_tags shape: {df_tags.shape}")
print(f"df_merged_data shape: {df_merged_data.shape}")
print(f"user_movie_matrix shape: {user_movie_matrix.shape}")

In [ ]:
import pandas as pd
import os
import re
import zipfile # Re-import zipfile if the notebook was completely reset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

# Re-initialize extraction_path
extraction_path = 'ml-32m'

# Open the zip file and extract its contents (if not already done in the new session)
# This is a safeguard, as runtime restart clears everything. Assuming ml-32m.zip is in /content/
if not os.path.exists(os.path.join(extraction_path, 'ml-32m')):
    print("Dataset not found in expected path, attempting extraction.")
    os.makedirs(extraction_path, exist_ok=True)
    with zipfile.ZipFile('/content/ml-32m.zip', 'r') as zip_ref:
        zip_ref.extractall(extraction_path)
    print(f"Dataset re-extracted to '{extraction_path}' directory.")
else:
    print(f"Dataset already extracted to '{extraction_path}' directory.")

# Load movies.csv and preprocess it
df_movies = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'movies.csv'))

df_movies['year'] = df_movies['title'].str.extract(r'\((\d{4})\)')
df_movies['year'] = pd.to_numeric(df_movies['year'], errors='coerce').fillna(0).astype(int)
df_movies['title'] = df_movies['title'].apply(lambda x: re.sub(r'\s*\(\d{4}\)$', '', x)).str.strip()
df_movies['genres'] = df_movies['genres'].apply(lambda x: x.split('|'))
df_movies['genres_str'] = df_movies['genres'].apply(lambda x: ' '.join(x))

# Recalculate TF-IDF matrix and cosine similarity for content-based component
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix_genres = tfidf_vectorizer.fit_transform(df_movies['genres_str'])
cosine_sim_genres = cosine_similarity(tfidf_matrix_genres, tfidf_matrix_genres)

# Load ratings.csv and preprocess it
df_ratings = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'ratings.csv'))
df_ratings.drop('timestamp', axis=1, inplace=True)

# Load links.csv and preprocess it (handle tmdbId missing values)
df_links = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'links.csv'))
df_links['tmdbId'] = df_links['tmdbId'].fillna(0).astype(int)

# Load tags.csv and preprocess it (drop rows with missing tags)
df_tags = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'tags.csv'))
df_tags.dropna(subset=['tag'], inplace=True)
df_tags.drop('timestamp', axis=1, inplace=True)

# Merge df_ratings with df_movies to recreate df_merged_data
df_merged_data = pd.merge(df_ratings, df_movies, on='movieId', how='left')

# Re-create user_movie_matrix for collaborative filtering
# Using a small sample due to potential memory issues on large dataset pivot_table
# Acknowledging the PerformanceWarning from previous attempts. For the SVD implementation,
# we might use a different representation or a subset if memory becomes an issue.
user_movie_matrix = df_merged_data.pivot_table(index='userId', columns='movieId', values='rating')

print("All necessary dataframes (df_movies, df_ratings, df_links, df_tags, df_merged_data, user_movie_matrix) have been reloaded and preprocessed.")
print(f"df_movies shape: {df_movies.shape}")
print(f"df_ratings shape: {df_ratings.shape}")
print(f"df_links shape: {df_links.shape}")
print(f"df_tags shape: {df_tags.shape}")
print(f"df_merged_data shape: {df_merged_data.shape}")
print(f"user_movie_matrix shape: {user_movie_matrix.shape}")

In [ ]:
import pandas as pd
import os
import re
import zipfile
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

# Re-initialize extraction_path
extraction_path = 'ml-32m'

# Open the zip file and extract its contents (if not already done in the new session)
# This is a safeguard, as runtime restart clears everything. Assuming ml-32m.zip is in /content/
if not os.path.exists(os.path.join(extraction_path, 'ml-32m')):
    print("Dataset not found in expected path, attempting extraction.")
    os.makedirs(extraction_path, exist_ok=True)
    with zipfile.ZipFile('/content/ml-32m.zip', 'r') as zip_ref:
        zip_ref.extractall(extraction_path)
    print(f"Dataset re-extracted to '{extraction_path}' directory.")
else:
    print(f"Dataset already extracted to '{extraction_path}' directory.")

# Load movies.csv and preprocess it
df_movies = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'movies.csv'))

df_movies['year'] = df_movies['title'].str.extract(r'\((\d{4})\)')
df_movies['year'] = pd.to_numeric(df_movies['year'], errors='coerce').fillna(0).astype(int)
df_movies['title'] = df_movies['title'].apply(lambda x: re.sub(r'\s*\(\d{4}\)$', '', x)).str.strip()
df_movies['genres'] = df_movies['genres'].apply(lambda x: x.split('|'))
df_movies['genres_str'] = df_movies['genres'].apply(lambda x: ' '.join(x))

# Recalculate TF-IDF matrix and cosine similarity for content-based component
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix_genres = tfidf_vectorizer.fit_transform(df_movies['genres_str'])
cosine_sim_genres = cosine_similarity(tfidf_matrix_genres, tfidf_matrix_genres)

# Load ratings.csv and preprocess it
df_ratings = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'ratings.csv'))
df_ratings.drop('timestamp', axis=1, inplace=True)

# Load links.csv and preprocess it (handle tmdbId missing values)
df_links = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'links.csv'))
df_links['tmdbId'] = df_links['tmdbId'].fillna(0).astype(int)

# Load tags.csv and preprocess it (drop rows with missing tags)
df_tags = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'tags.csv'))
df_tags.dropna(subset=['tag'], inplace=True)
df_tags.drop('timestamp', axis=1, inplace=True)

# Merge df_ratings with df_movies to recreate df_merged_data
df_merged_data = pd.merge(df_ratings, df_movies, on='movieId', how='left')

# Re-create user_movie_matrix for collaborative filtering
# Using a small sample due to potential memory issues on large dataset pivot_table
# Acknowledging the PerformanceWarning from previous attempts. For the SVD implementation,
# we might use a different representation or a subset if memory becomes an issue.
user_movie_matrix = df_merged_data.pivot_table(index='userId', columns='movieId', values='rating')

print("All necessary dataframes (df_movies, df_ratings, df_links, df_tags, df_merged_data, user_movie_matrix) have been reloaded and preprocessed.")
print(f"df_movies shape: {df_movies.shape}")
print(f"df_ratings shape: {df_ratings.shape}")
print(f"df_links shape: {df_links.shape}")
print(f"df_tags shape: {df_tags.shape}")
print(f"df_merged_data shape: {df_merged_data.shape}")
print(f"user_movie_matrix shape: {user_movie_matrix.shape}")

In [ ]:
import pandas as pd
import os
import re
import zipfile
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

# Re-initialize extraction_path
extraction_path = 'ml-32m'

# Open the zip file and extract its contents (if not already done in the new session)
# This is a safeguard, as runtime restart clears everything. Assuming ml-32m.zip is in /content/
if not os.path.exists(os.path.join(extraction_path, 'ml-32m')):
    print("Dataset not found in expected path, attempting extraction.")
    os.makedirs(extraction_path, exist_ok=True)
    with zipfile.ZipFile('/content/ml-32m.zip', 'r') as zip_ref:
        zip_ref.extractall(extraction_path)
    print(f"Dataset re-extracted to '{extraction_path}' directory.")
else:
    print(f"Dataset already extracted to '{extraction_path}' directory.")

# Load movies.csv and preprocess it
df_movies = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'movies.csv'))

df_movies['year'] = df_movies['title'].str.extract(r'\((\d{4})\)')
df_movies['year'] = pd.to_numeric(df_movies['year'], errors='coerce').fillna(0).astype(int)
df_movies['title'] = df_movies['title'].apply(lambda x: re.sub(r'\s*\(\d{4}\)$', '', x)).str.strip()
df_movies['genres'] = df_movies['genres'].apply(lambda x: x.split('|'))
df_movies['genres_str'] = df_movies['genres'].apply(lambda x: ' '.join(x))

# Recalculate TF-IDF matrix and cosine similarity for content-based component
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix_genres = tfidf_vectorizer.fit_transform(df_movies['genres_str'])
cosine_sim_genres = cosine_similarity(tfidf_matrix_genres, tfidf_matrix_genres)

# Load ratings.csv and preprocess it
df_ratings = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'ratings.csv'))
df_ratings.drop('timestamp', axis=1, inplace=True)

# Load links.csv and preprocess it (handle tmdbId missing values)
df_links = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'links.csv'))
df_links['tmdbId'] = df_links['tmdbId'].fillna(0).astype(int)

# Load tags.csv and preprocess it (drop rows with missing tags)
df_tags = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'tags.csv'))
df_tags.dropna(subset=['tag'], inplace=True)
df_tags.drop('timestamp', axis=1, inplace=True)

# Merge df_ratings with df_movies to recreate df_merged_data
df_merged_data = pd.merge(df_ratings, df_movies, on='movieId', how='left')

# Re-create user_movie_matrix for collaborative filtering
# Using a small sample due to potential memory issues on large dataset pivot_table
# Acknowledging the PerformanceWarning from previous attempts. For the SVD implementation,
# we might use a different representation or a subset if memory becomes an issue.
user_movie_matrix = df_merged_data.pivot_table(index='userId', columns='movieId', values='rating')

print("All necessary dataframes (df_movies, df_ratings, df_links, df_tags, df_merged_data, user_movie_matrix) have been reloaded and preprocessed.")
print(f"df_movies shape: {df_movies.shape}")
print(f"df_ratings shape: {df_ratings.shape}")
print(f"df_links shape: {df_links.shape}")
print(f"df_tags shape: {df_tags.shape}")
print(f"df_merged_data shape: {df_merged_data.shape}")
print(f"user_movie_matrix shape: {user_movie_matrix.shape}")

In [ ]:
import pandas as pd
import os
import re
import zipfile
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

# Re-initialize extraction_path
extraction_path = 'ml-32m'

# Open the zip file and extract its contents (if not already done in the new session)
# This is a safeguard, as runtime restart clears everything. Assuming ml-32m.zip is in /content/
if not os.path.exists(os.path.join(extraction_path, 'ml-32m')):
    print("Dataset not found in expected path, attempting extraction.")
    os.makedirs(extraction_path, exist_ok=True)
    with zipfile.ZipFile('/content/ml-32m.zip', 'r') as zip_ref:
        zip_ref.extractall(extraction_path)
    print(f"Dataset re-extracted to '{extraction_path}' directory.")
else:
    print(f"Dataset already extracted to '{extraction_path}' directory.")

# Load movies.csv and preprocess it
df_movies = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'movies.csv'))

df_movies['year'] = df_movies['title'].str.extract(r'\((\d{4})\)')
df_movies['year'] = pd.to_numeric(df_movies['year'], errors='coerce').fillna(0).astype(int)
df_movies['title'] = df_movies['title'].apply(lambda x: re.sub(r'\s*\(\d{4}\)$', '', x)).str.strip()
df_movies['genres'] = df_movies['genres'].apply(lambda x: x.split('|'))
df_movies['genres_str'] = df_movies['genres'].apply(lambda x: ' '.join(x))

# Recalculate TF-IDF matrix and cosine similarity for content-based component
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix_genres = tfidf_vectorizer.fit_transform(df_movies['genres_str'])
cosine_sim_genres = cosine_similarity(tfidf_matrix_genres, tfidf_matrix_genres)

# Load ratings.csv and preprocess it
df_ratings = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'ratings.csv'))
df_ratings.drop('timestamp', axis=1, inplace=True)

# Load links.csv and preprocess it (handle tmdbId missing values)
df_links = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'links.csv'))
df_links['tmdbId'] = df_links['tmdbId'].fillna(0).astype(int)

# Load tags.csv and preprocess it (drop rows with missing tags)
df_tags = pd.read_csv(os.path.join(extraction_path, 'ml-32m', 'tags.csv'))
df_tags.dropna(subset=['tag'], inplace=True)
df_tags.drop('timestamp', axis=1, inplace=True)

# Merge df_ratings with df_movies to recreate df_merged_data
df_merged_data = pd.merge(df_ratings, df_movies, on='movieId', how='left')

# Re-create user_movie_matrix for collaborative filtering
# Using a small sample due to potential memory issues on large dataset pivot_table
# Acknowledging the PerformanceWarning from previous attempts. For the SVD implementation,
# we might use a different representation or a subset if memory becomes an issue.
user_movie_matrix = df_merged_data.pivot_table(index='userId', columns='movieId', values='rating')

print("All necessary dataframes (df_movies, df_ratings, df_links, df_tags, df_merged_data, user_movie_matrix) have been reloaded and preprocessed.")
print(f"df_movies shape: {df_movies.shape}")
print(f"df_ratings shape: {df_ratings.shape}")
print(f"df_links shape: {df_links.shape}")
print(f"df_tags shape: {df_tags.shape}")
print(f"df_merged_data shape: {df_merged_data.shape}")
print(f"user_movie_matrix shape: {user_movie_matrix.shape}")

In [ ]:
import pandas as pd
import os
import re
import zipfile
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Extraction
extraction_path = 'ml-32m'
os.makedirs(extraction_path, exist_ok=True)
with zipfile.ZipFile('/content/ml-32m.zip', 'r') as zip_ref:
    zip_ref.extractall(extraction_path)
print('Dataset extracted successfully.')

# 2. Loading
base_dir = os.path.join(extraction_path, 'ml-32m')
df_movies = pd.read_csv(os.path.join(base_dir, 'movies.csv'))
df_ratings = pd.read_csv(os.path.join(base_dir, 'ratings.csv'))
df_links = pd.read_csv(os.path.join(base_dir, 'links.csv'))
df_tags = pd.read_csv(os.path.join(base_dir, 'tags.csv'))

# 3. Cleaning movies
df_movies['year'] = df_movies['title'].str.extract(r'\((\d{4})\)')
df_movies['year'] = pd.to_numeric(df_movies['year'], errors='coerce').fillna(0).astype(int)
df_movies['title'] = df_movies['title'].apply(lambda x: re.sub(r'\s*\(\d{4}\)$', '', x)).str.strip()
df_movies['genres_str'] = df_movies['genres'].str.replace('|', ' ', regex=False)

# 4. Preprocessing other DFs
df_ratings.drop('timestamp', axis=1, inplace=True)
df_tags.drop('timestamp', axis=1, inplace=True)
df_tags.dropna(subset=['tag'], inplace=True)
df_links['tmdbId'] = df_links['tmdbId'].fillna(0).astype(int)

# 5. Merging
df_merged_data = pd.merge(df_ratings, df_movies, on='movieId', how='left')

# 6. User-Movie Matrix (Subset to avoid memory error)
# Using first 1,000,000 rows as requested
user_movie_matrix = df_ratings.iloc[:1000000].pivot_table(index='userId', columns='movieId', values='rating')

# 7. TF-IDF and Cosine Similarity
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_movies['genres_str'])
cosine_sim_genres = cosine_similarity(tfidf_matrix, tfidf_matrix)

print('Environment re-initialized.')
print(f'Movies: {df_movies.shape}, Ratings: {df_ratings.shape}, Merged: {df_merged_data.shape}')
print(f'Matrix shape: {user_movie_matrix.shape}')

In [ ]:
import pandas as pd
import os
import re
import zipfile
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Extraction
extraction_path = 'ml-32m'
os.makedirs(extraction_path, exist_ok=True)
with zipfile.ZipFile('/content/ml-32m.zip', 'r') as zip_ref:
    zip_ref.extractall(extraction_path)
print('Dataset extracted successfully.')

# 2. Loading
base_dir = os.path.join(extraction_path, 'ml-32m')
df_movies = pd.read_csv(os.path.join(base_dir, 'movies.csv'))
df_ratings = pd.read_csv(os.path.join(base_dir, 'ratings.csv'))
df_links = pd.read_csv(os.path.join(base_dir, 'links.csv'))
df_tags = pd.read_csv(os.path.join(base_dir, 'tags.csv'))

# 3. Preprocess df_movies
df_movies['year'] = df_movies['title'].str.extract(r'\((\d{4})\)')
df_movies['year'] = pd.to_numeric(df_movies['year'], errors='coerce').fillna(0).astype(int)
df_movies['title'] = df_movies['title'].apply(lambda x: re.sub(r'\s*\(\d{4}\)$', '', x)).str.strip()
df_movies['genres_str'] = df_movies['genres'].str.replace('|', ' ', regex=False)

# 4. Preprocess other DataFrames
df_ratings.drop('timestamp', axis=1, inplace=True)
df_tags.drop('timestamp', axis=1, inplace=True)
df_tags.dropna(subset=['tag'], inplace=True)
df_links['tmdbId'] = df_links['tmdbId'].fillna(0).astype(int)

# 5. Merge df_ratings with df_movies
df_merged_data = pd.merge(df_ratings, df_movies, on='movieId', how='left')

# 6. Create user_movie_matrix (subset to manage memory)
user_movie_matrix = df_ratings.iloc[:1000000].pivot_table(index='userId', columns='movieId', values='rating')

# 7. Content-based feature engineering (TF-IDF & Cosine Similarity)
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_movies['genres_str'])
cosine_sim_genres = cosine_similarity(tfidf_matrix, tfidf_matrix)

print('System Initialization Complete.')
print(f'Movies shape: {df_movies.shape}')
print(f'Ratings shape: {df_ratings.shape}')
print(f'User-Movie Matrix subset shape: {user_movie_matrix.shape}')
print(f'Cosine Similarity Matrix shape: {cosine_sim_genres.shape}')

In [ ]:
import pandas as pd
import os

import re
import zipfile
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 2. Define path and extract zip file
zip_path = '/content/ml-32m.zip'
extraction_path = 'ml-32m'
os.makedirs(extraction_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)
print('Dataset extracted successfully.')

# 3. Load DataFrames
base_dir = os.path.join(extraction_path, 'ml-32m')
df_movies = pd.read_csv(os.path.join(base_dir, 'movies.csv'))
df_ratings = pd.read_csv(os.path.join(base_dir, 'ratings.csv'))
df_links = pd.read_csv(os.path.join(base_dir, 'links.csv'))
df_tags = pd.read_csv(os.path.join(base_dir, 'tags.csv'))

# 4. Clean df_movies
df_movies['year'] = df_movies['title'].str.extract(r'\((\d{4})\)')
df_movies['year'] = pd.to_numeric(df_movies['year'], errors='coerce').fillna(0).astype(int)
df_movies['title'] = df_movies['title'].apply(lambda x: re.sub(r'\s*\(\d{4}\)$', '', x)).str.strip()
df_movies['genres_str'] = df_movies['genres'].str.replace('|', ' ', regex=False)

# 5. Preprocess other DataFrames
df_ratings.drop('timestamp', axis=1, inplace=True)
df_tags.drop('timestamp', axis=1, inplace=True)
df_tags.dropna(subset=['tag'], inplace=True)
df_links['tmdbId'] = df_links['tmdbId'].fillna(0).astype(int)

# 6. Merge DataFrames
df_merged_data = pd.merge(df_ratings, df_movies, on='movieId', how='left')

# 7. Content-based Feature Engineering
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_movies['genres_str'])
cosine_sim_genres = cosine_similarity(tfidf_matrix, tfidf_matrix)

print('System Initialization and Feature Engineering Complete.')
print(f'Movies: {df_movies.shape}, Ratings: {df_ratings.shape}')
print(f'TF-IDF Matrix Shape: {tfidf_matrix.shape}')

In [ ]:
import pandas as pd
import os
import re
import zipfile
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 2. Extract contents
zip_path = '/content/ml-32m.zip'
extraction_path = 'ml-32m'
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)

# 3. Load DataFrames
base_dir = os.path.join(extraction_path, 'ml-32m')
df_movies = pd.read_csv(os.path.join(base_dir, 'movies.csv'))
df_ratings = pd.read_csv(os.path.join(base_dir, 'ratings.csv'))
df_links = pd.read_csv(os.path.join(base_dir, 'links.csv'))
df_tags = pd.read_csv(os.path.join(base_dir, 'tags.csv'))

# 4. Clean df_movies
df_movies['year'] = df_movies['title'].str.extract(r'\((\d{4})\)')
df_movies['year'] = pd.to_numeric(df_movies['year'], errors='coerce').fillna(0).astype(int)
df_movies['title'] = df_movies['title'].apply(lambda x: re.sub(r'\s*\(\d{4}\)$', '', x)).str.strip()
df_movies['genres_str'] = df_movies['genres'].str.replace('|', ' ', regex=False)

# 5. Preprocess other DataFrames
df_ratings.drop('timestamp', axis=1, inplace=True)
df_tags.drop('timestamp', axis=1, inplace=True)
df_tags.dropna(subset=['tag'], inplace=True)
df_links['tmdbId'] = df_links['tmdbId'].fillna(0).astype(int)

# 6. Merge DataFrames
df_merged_data = pd.merge(df_ratings, df_movies, on='movieId', how='left')

# 7. Initialize TfidfVectorizer and create tfidf_matrix
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_movies['genres_str'])

# 8. Compute cosine similarity matrix
cosine_sim_genres = cosine_similarity(tfidf_matrix, tfidf_matrix)

print('Data loading, cleaning, and content-based matrix calculation complete.')
print(f'Merged data shape: {df_merged_data.shape}')
print(f'Cosine similarity matrix shape: {cosine_sim_genres.shape}')

In [ ]:
### Action Required: Factory Reset Runtime

The persistent `ValueError: numpy.dtype size changed` indicates a critical binary incompatibility between the installed versions of `pandas` and `numpy` within this environment. This specific issue cannot be resolved through standard package reinstallation or simple runtime restarts.

To stabilize the environment and proceed with building the recommender system, please perform a **'Factory reset runtime'**:
1. Go to the **Runtime** menu in the top toolbar of Google Colab.
2. Select **'Factory reset runtime'**.
3. Confirm the reset and wait for the notebook to reconnect.

**Note:** This action will clear all currently defined variables, local files, and installed packages. This is a necessary step to ensure we start with a clean, compatible environment. Once the reset is complete, please proceed to re-execute the data initialization block.

In [ ]:
from scipy.sparse import csr_matrix

# Map userId and movieId to contiguous categorical codes for matrix indexing
df_ratings['user_cat'] = df_ratings['userId'].astype('category')
df_ratings['movie_cat'] = df_ratings['movieId'].astype('category')

# Create the sparse CSR matrix
# data: ratings values, row: user codes, col: movie codes
user_item_sparse = csr_matrix((
    df_ratings['rating'],
    (df_ratings['user_cat'].cat.codes, df_ratings['movie_cat'].cat.codes)
))

# Calculate sparsity
n_users = user_item_sparse.shape[0]
n_items = user_item_sparse.shape[1]
possible_entries = n_users * n_items
sparsity = (1 - (user_item_sparse.nnz / possible_entries)) * 100

print(f'Sparse Matrix Shape: {user_item_sparse.shape}')
print(f'Number of non-zero entries (ratings): {user_item_sparse.nnz}')
print(f'Sparsity: {sparsity:.4f}%')
print('Sparse matrix implementation complete.')

In [ ]:
import numpy as np
from sklearn.decomposition import TruncatedSVD

class HybridRecommender:
    def __init__(self, df_movies, df_ratings, user_item_sparse):
        self.df_movies = df_movies.copy()
        self.df_ratings = df_ratings
        self.user_item_sparse = user_item_sparse

        # Initialize placeholders
        self.tfidf_vectorizer = TfidfVectorizer(stop_words='english')
        self.tfidf_matrix = None
        self.svd_model = TruncatedSVD(n_components=20, random_state=42)
        self.matrix_factorized = None

    def fit(self):
        print('Fitting Content-Based Component (TF-IDF)...')
        self.tfidf_matrix = self.tfidf_vectorizer.fit_transform(self.df_movies['genres_str'])

        print('Fitting Collaborative Component (SVD)...')
        self.matrix_factorized = self.svd_model.fit_transform(self.user_item_sparse)
        print('Model training complete.')

    def get_collaborative_recommendations(self, user_id, top_n=100):
        # Map raw userId to categorical index
        if user_id not in self.df_ratings['userId'].values:
            return []

        user_idx = self.df_ratings[self.df_ratings['userId'] == user_id]['user_cat'].cat.codes.iloc[0]

        # Predict ratings: dot product of user latent factors and item components
        predicted_ratings = np.dot(self.matrix_factorized[user_idx, :], self.svd_model.components_)

        # Get top movie indices from the prediction
        movie_indices = np.argsort(predicted_ratings)[::-1][:top_n]

        # Map categorical indices back to raw movieIds
        movie_categories = self.df_ratings['movie_cat'].cat.categories
        recommended_movie_ids = [movie_categories[i] for i in movie_indices]

        return recommended_movie_ids

    def get_hybrid_recommendations(self, user_id, movie_title, top_n=10):
        if movie_title not in self.df_movies['title'].values:
            return 'Movie not found.'

        # 1. Content-based scores
        movie_idx = self.df_movies[self.df_movies['title'] == movie_title].index[0]
        content_sim_scores = cosine_similarity(self.tfidf_matrix[movie_idx], self.tfidf_matrix).flatten()

        # 2. Collaborative candidates
        collab_movie_ids = self.get_collaborative_recommendations(user_id, top_n=200)

        # 3. Hybridization with boost
        hybrid_results = []
        for i, score in enumerate(content_sim_scores):
            m_id = self.df_movies.iloc[i]['movieId']
            if self.df_movies.iloc[i]['title'] == movie_title: continue

            final_score = score
            if m_id in collab_movie_ids:
                final_score += 0.5  # Additive boost for collaborative agreement

            hybrid_results.append((m_id, final_score))

        # Sort and fetch top N
        hybrid_results = sorted(hybrid_results, key=lambda x: x[1], reverse=True)[:top_n]
        top_ids = [x[0] for x in hybrid_results]

        return self.df_movies[self.df_movies['movieId'].isin(top_ids)][['title', 'genres_str']]

# Initialize and train the engine
recommender = HybridRecommender(df_movies, df_ratings, user_item_sparse)
recommender.fit()

print('\nGenerating Hybrid Recommendations for User 1 based on "Toy Story":')
display(recommender.get_hybrid_recommendations(user_id=1, movie_title='Toy Story', top_n=5))

In [ ]:
import joblib
import os

# 1. Extend the HybridRecommender class with persistence methods
def save_model(self, path='models'):
    if not os.path.exists(path):
        os.makedirs(path)

    joblib.dump(self.svd_model, os.path.join(path, 'svd_model.pkl'))
    joblib.dump(self.tfidf_vectorizer, os.path.join(path, 'tfidf_vectorizer.pkl'))
    joblib.dump(self.tfidf_matrix, os.path.join(path, 'tfidf_matrix.pkl'))
    joblib.dump(self.matrix_factorized, os.path.join(path, 'matrix_factorized.pkl'))
    print(f'Model components saved to {path}/')

def load_model(self, path='models'):
    self.svd_model = joblib.load(os.path.join(path, 'svd_model.pkl'))
    self.tfidf_vectorizer = joblib.load(os.path.join(path, 'tfidf_vectorizer.pkl'))
    self.tfidf_matrix = joblib.load(os.path.join(path, 'tfidf_matrix.pkl'))
    self.matrix_factorized = joblib.load(os.path.join(path, 'matrix_factorized.pkl'))
    print(f'Model components loaded from {path}/')

# Attach methods to the class dynamically
HybridRecommender.save_model = save_model
HybridRecommender.load_model = load_model

# 2. Save the currently trained recommender
recommender.save_model('models')

# 3. Demonstrate loading into a new instance (without fitting)
print('\nInitializing a fresh recommender instance...')
production_recommender = HybridRecommender(df_movies, df_ratings, user_item_sparse)
production_recommender.load_model('models')

# 4. Verify functionality
print('\nGenerating recommendations from the LOADED production engine:')
display(production_recommender.get_hybrid_recommendations(user_id=1, movie_title='Toy Story', top_n=5))

In [ ]:
import numpy as np

# Update methods in the existing HybridRecommender class with validation and error handling
def get_collaborative_recommendations_v2(self, user_id, top_n=100):
    """Gets SVD predictions with user validation and cold-start support."""
    try:
        # Check if user exists in the rating dataset
        if user_id not in self.df_ratings['userId'].values:
            print(f'User ID {user_id} not found (Cold Start). Returning empty collaborative list.')
            return []

        user_idx = self.df_ratings[self.df_ratings['userId'] == user_id]['user_cat'].cat.codes.iloc[0]

        # Predict ratings
        predicted_ratings = np.dot(self.matrix_factorized[user_idx, :], self.svd_model.components_)
        movie_indices = np.argsort(predicted_ratings)[::-1][:top_n]

        movie_categories = self.df_ratings['movie_cat'].cat.categories
        recommended_movie_ids = [movie_categories[i] for i in movie_indices]

        return recommended_movie_ids
    except Exception as e:
        print(f'Error in collaborative filtering: {e}')
        return []

def get_hybrid_recommendations_v2(self, user_id, movie_title, top_n=10):
    """Hybrid recommendations with input validation and graceful fallback."""
    try:
        # 1. Validate Movie Title
        if movie_title not in self.df_movies['title'].values:
            return 'Movie not found in database.'

        # 2. Content-based scores
        movie_idx = self.df_movies[self.df_movies['title'] == movie_title].index[0]
        content_sim_scores = cosine_similarity(self.tfidf_matrix[movie_idx], self.tfidf_matrix).flatten()

        # 3. Collaborative candidates (handles Cold Start internally via empty list)
        collab_movie_ids = self.get_collaborative_recommendations(user_id, top_n=200)

        # 4. Hybridization logic
        hybrid_results = []
        for i, score in enumerate(content_sim_scores):
            m_id = self.df_movies.iloc[i]['movieId']
            if self.df_movies.iloc[i]['title'] == movie_title:
                continue

            final_score = score
            # Apply boost only if collaborative data exists for this user
            if collab_movie_ids and (m_id in collab_movie_ids):
                final_score += 0.5

            hybrid_results.append((m_id, final_score))

        hybrid_results = sorted(hybrid_results, key=lambda x: x[1], reverse=True)[:top_n]
        top_ids = [x[0] for x in hybrid_results]

        return self.df_movies[self.df_movies['movieId'].isin(top_ids)][['title', 'genres_str']]

    except (IndexError, ValueError) as e:
        return f'An error occurred while processing indices or values: {e}'
    except Exception as e:
        return f'An unexpected error occurred: {e}'

# Inject updated methods into the existing instance and class
HybridRecommender.get_collaborative_recommendations = get_collaborative_recommendations_v2
HybridRecommender.get_hybrid_recommendations = get_hybrid_recommendations_v2

print('Testing validation with non-existent User ID (999999):')
test_user = production_recommender.get_hybrid_recommendations(user_id=999999, movie_title='Toy Story', top_n=3)
display(test_user)

print('\nTesting validation with fake Movie Title ("The Matrix 5"):')
test_movie = production_recommender.get_hybrid_recommendations(user_id=1, movie_title='The Matrix 5', top_n=3)
print(f'Result: {test_movie}')

In [5]:
import os
import re
import zipfile
import pandas as pd

# 2. Define path and extract zip file
zip_path = '/content/ml-32m.zip'
extraction_path = 'ml-32m'
os.makedirs(extraction_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)
print('Dataset extracted successfully.')

# 3. Load DataFrames
base_dir = os.path.join(extraction_path, 'ml-32m')
df_movies = pd.read_csv(os.path.join(base_dir, 'movies.csv'))
df_ratings = pd.read_csv(os.path.join(base_dir, 'ratings.csv'))

# 4. Preprocess df_movies
df_movies['year'] = df_movies['title'].str.extract(r'\((\d{4})\)')
df_movies['year'] = pd.to_numeric(df_movies['year'], errors='coerce').fillna(0).astype(int)
df_movies['title'] = df_movies['title'].apply(lambda x: re.sub(r'\s*\(\d{4}\)$', '', x)).str.strip()
df_movies['genres_str'] = df_movies['genres'].str.replace('|', ' ', regex=False)

# 5. Preprocess df_ratings
df_ratings.drop('timestamp', axis=1, inplace=True, errors='ignore')

# 6. Verify results
print(f'\ndf_movies head (Shape: {df_movies.shape}):')
display(df_movies.head())

print(f'\ndf_ratings head (Shape: {df_ratings.shape}):')
display(df_ratings.head())

Dataset extracted successfully.

df_movies head (Shape: (87585, 5)):


,movieId,title,genres,year,genres_str
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995,Adventure Animation Children Comedy Fantasy
1,2,Jumanji,Adventure|Children|Fantasy,1995,Adventure Children Fantasy
2,3,Grumpier Old Men,Comedy|Romance,1995,Comedy Romance
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995,Comedy Drama Romance
4,5,Father of the Bride Part II,Comedy,1995,Comedy



df_ratings head (Shape: (32000204, 3)):


,userId,movieId,rating
0,1,17,4.0
1,1,25,1.0
2,1,29,2.0
3,1,30,5.0
4,1,32,5.0


In [6]:
from scipy.sparse import csr_matrix
import numpy as np

# 1. Convert IDs to categorical data type for efficient indexing
df_ratings['user_cat'] = df_ratings['userId'].astype('category')
df_ratings['movie_cat'] = df_ratings['movieId'].astype('category')

# 2. Create the sparse CSR matrix
# row indices: user categorical codes, col indices: movie categorical codes, data: ratings
user_item_sparse = csr_matrix((
    df_ratings['rating'],
    (df_ratings['user_cat'].cat.codes, df_ratings['movie_cat'].cat.codes)
))

# 3. Calculate sparsity and print diagnostics
n_users, n_items = user_item_sparse.shape
possible_entries = n_users * n_items
sparsity = (1 - (user_item_sparse.nnz / possible_entries)) * 100

print(f'Sparse Matrix Shape: {user_item_sparse.shape}')
print(f'Number of non-zero entries (ratings): {user_item_sparse.nnz}')
print(f'Sparsity: {sparsity:.4f}%')
print('Sparse matrix reconstruction complete.')

Sparse Matrix Shape: (200948, 84432)
Number of non-zero entries (ratings): 32000204
Sparsity: 99.8114%
Sparse matrix reconstruction complete.


In [13]:
# Execute the evaluation function on the production recommender
final_rmse = evaluate_model(recommender)
print(f'\nModel Accuracy Analysis complete. RMSE: {final_rmse:.4f}')

Preparing Evaluation...
Evaluation Results:
RMSE: 2.6272

Model Accuracy Analysis complete. RMSE: 2.6272


In [7]:
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

class HybridRecommender:
    def __init__(self, df_movies, df_ratings, user_item_sparse):
        self.df_movies = df_movies.copy()
        self.df_ratings = df_ratings
        self.user_item_sparse = user_item_sparse

        # Initialize placeholders
        self.tfidf_vectorizer = TfidfVectorizer(stop_words='english')
        self.tfidf_matrix = None
        self.svd_model = TruncatedSVD(n_components=20, random_state=42)
        self.matrix_factorized = None

    def fit(self):
        print('Fitting Content-Based Component (TF-IDF)...')
        self.tfidf_matrix = self.tfidf_vectorizer.fit_transform(self.df_movies['genres_str'])

        print('Fitting Collaborative Component (SVD)...')
        self.matrix_factorized = self.svd_model.fit_transform(self.user_item_sparse)
        print('Model training complete.')

    def get_collaborative_recommendations(self, user_id, top_n=100):
        if user_id not in self.df_ratings['userId'].values:
            return []

        # Map raw userId to categorical index
        user_idx = self.df_ratings[self.df_ratings['userId'] == user_id]['user_cat'].cat.codes.iloc[0]

        # Predict ratings: dot product of user latent factors and item components
        predicted_ratings = np.dot(self.matrix_factorized[user_idx, :], self.svd_model.components_)

        # Get top movie indices from the prediction
        movie_indices = np.argsort(predicted_ratings)[::-1][:top_n]

        # Map categorical indices back to raw movieIds
        movie_categories = self.df_ratings['movie_cat'].cat.categories
        recommended_movie_ids = [movie_categories[i] for i in movie_indices]

        return recommended_movie_ids

    def get_hybrid_recommendations(self, user_id, movie_title, top_n=10):
        if movie_title not in self.df_movies['title'].values:
            return 'Movie not found.'

        # 1. Content-based scores
        movie_idx = self.df_movies[self.df_movies['title'] == movie_title].index[0]
        content_sim_scores = cosine_similarity(self.tfidf_matrix[movie_idx], self.tfidf_matrix).flatten()

        # 2. Collaborative candidates
        collab_movie_ids = self.get_collaborative_recommendations(user_id, top_n=200)

        # 3. Hybridization with boost
        hybrid_results = []
        for i, score in enumerate(content_sim_scores):
            m_id = self.df_movies.iloc[i]['movieId']
            if self.df_movies.iloc[i]['title'] == movie_title: continue

            final_score = score
            if m_id in collab_movie_ids:
                final_score += 0.5  # Additive boost for collaborative agreement

            hybrid_results.append((m_id, final_score))

        # Sort and fetch top N
        hybrid_results = sorted(hybrid_results, key=lambda x: x[1], reverse=True)[:top_n]
        top_ids = [x[0] for x in hybrid_results]

        return self.df_movies[self.df_movies['movieId'].isin(top_ids)][['title', 'genres_str']]

print('HybridRecommender class defined.')

HybridRecommender class defined.


In [8]:
import joblib
import os

# 1. Define standalone functions for saving and loading
def save_model(self, path='models'):
    if not os.path.exists(path):
        os.makedirs(path)

    joblib.dump(self.svd_model, os.path.join(path, 'svd_model.pkl'))
    joblib.dump(self.tfidf_vectorizer, os.path.join(path, 'tfidf_vectorizer.pkl'))
    joblib.dump(self.tfidf_matrix, os.path.join(path, 'tfidf_matrix.pkl'))
    joblib.dump(self.matrix_factorized, os.path.join(path, 'matrix_factorized.pkl'))
    print(f'Model components successfully saved to {path}/')

def load_model(self, path='models'):
    self.svd_model = joblib.load(os.path.join(path, 'svd_model.pkl'))
    self.tfidf_vectorizer = joblib.load(os.path.join(path, 'tfidf_vectorizer.pkl'))
    self.tfidf_matrix = joblib.load(os.path.join(path, 'tfidf_matrix.pkl'))
    self.matrix_factorized = joblib.load(os.path.join(path, 'matrix_factorized.pkl'))
    print(f'Model components successfully loaded from {path}/')

# 2. Dynamically attach functions as methods to the HybridRecommender class
HybridRecommender.save_model = save_model
HybridRecommender.load_model = load_model

# 3. Instantiate and train the initial recommender
print('Initializing and training the initial recommender...')
recommender = HybridRecommender(df_movies, df_ratings, user_item_sparse)
recommender.fit()

# 4. Save the trained model to disk
recommender.save_model('models')

# 5. Create a production instance and load the saved components
print('\nInitializing production instance and loading saved model...')
production_recommender = HybridRecommender(df_movies, df_ratings, user_item_sparse)
production_recommender.load_model('models')

# 6. Verify persistence by generating recommendations for User 1 based on Toy Story
print('\nGenerating recommendations from the LOADED production engine for User 1 (Toy Story):')
recs = production_recommender.get_hybrid_recommendations(user_id=1, movie_title='Toy Story', top_n=5)
display(recs)

Initializing and training the initial recommender...
Fitting Content-Based Component (TF-IDF)...
Fitting Collaborative Component (SVD)...
Model training complete.
Model components successfully saved to models/

Initializing production instance and loading saved model...
Model components successfully loaded from models/

Generating recommendations from the LOADED production engine for User 1 (Toy Story):


,title,genres_str
729,Wallace & Gromit: A Close Shave,Animation Children Comedy
898,"Wizard of Oz, The",Adventure Children Fantasy Musical
1108,Monty Python and the Holy Grail,Adventure Comedy Fantasy
1120,Wallace & Gromit: The Wrong Trousers,Animation Children Comedy Crime
1167,"Princess Bride, The",Action Adventure Comedy Fantasy Romance


In [9]:
import os

# 1. Instantiate the HybridRecommender class
print('Initializing the HybridRecommender engine...')
final_recommender = HybridRecommender(df_movies, df_ratings, user_item_sparse)

# 2. Fit the models (TF-IDF and TruncatedSVD)
print('Fitting components on the full 32M ratings dataset...')
final_recommender.fit()

# 3. Final test for User ID 1 and 'Toy Story'
print('\n--- Test 1: Initial Trained Model ---')
print('Generating hybrid recommendations for User 1 based on "Toy Story":')
initial_recs = final_recommender.get_hybrid_recommendations(user_id=1, movie_title='Toy Story', top_n=5)
display(initial_recs)

# 4. Save the trained components to 'production_models'
model_dir = 'production_models'
final_recommender.save_model(model_dir)

# 5. Create a second instance and load the model state
print('\n--- Test 2: Persistent Production Engine ---')
print('Initializing a second instance and loading from saved state...')
production_engine = HybridRecommender(df_movies, df_ratings, user_item_sparse)
production_engine.load_model(model_dir)

# Demonstrate recommendations from the loaded state without re-running .fit()
print('Generating recommendations from the loaded engine for User 1:')
production_recs = production_engine.get_hybrid_recommendations(user_id=1, movie_title='Toy Story', top_n=5)
display(production_recs)

print('\nSystem Validation Complete: Production-ready engine is functional and persistent.')

Initializing the HybridRecommender engine...
Fitting components on the full 32M ratings dataset...
Fitting Content-Based Component (TF-IDF)...
Fitting Collaborative Component (SVD)...
Model training complete.

--- Test 1: Initial Trained Model ---
Generating hybrid recommendations for User 1 based on "Toy Story":


,title,genres_str
729,Wallace & Gromit: A Close Shave,Animation Children Comedy
898,"Wizard of Oz, The",Adventure Children Fantasy Musical
1108,Monty Python and the Holy Grail,Adventure Comedy Fantasy
1120,Wallace & Gromit: The Wrong Trousers,Animation Children Comedy Crime
1167,"Princess Bride, The",Action Adventure Comedy Fantasy Romance


Model components successfully saved to production_models/

--- Test 2: Persistent Production Engine ---
Initializing a second instance and loading from saved state...
Model components successfully loaded from production_models/
Generating recommendations from the loaded engine for User 1:


,title,genres_str
729,Wallace & Gromit: A Close Shave,Animation Children Comedy
898,"Wizard of Oz, The",Adventure Children Fantasy Musical
1108,Monty Python and the Holy Grail,Adventure Comedy Fantasy
1120,Wallace & Gromit: The Wrong Trousers,Animation Children Comedy Crime
1167,"Princess Bride, The",Action Adventure Comedy Fantasy Romance



System Validation Complete: Production-ready engine is functional and persistent.
